## Etapa 1. Carregamento dos dados

In [45]:
import pandas as pd

In [46]:
df = pd.read_csv("vendas_restaurante.csv")

In [47]:
df.head(10)

,id_pedido,data,hora,canal,item,categoria,quantidade,preco_unitario,valor_total,forma_pagamento
0,1068,2026-08-01,10:30,Salão,Refrigerante Lata,Bebidas,1,6.5,6.5,Cartão de Débito
1,1054,2026-08-01,10:56,Salão,Feijoada (sábado),Pratos,1,39.0,39.0,Cartão de Débito
2,1054,2026-08-01,10:56,Salão,Marmitex Tradicional,Pratos,2,22.0,44.0,Cartão de Débito
3,1054,2026-08-01,10:56,Salão,Refrigerante Lata,Bebidas,2,6.5,13.0,Cartão de Débito
4,1012,2026-08-01,11:01,Salão,Água Mineral,Bebidas,1,4.0,4.0,Cartão de Crédito
5,1037,2026-08-01,11:08,Salão,Feijoada (sábado),Pratos,2,39.0,78.0,Cartão de Crédito
6,1037,2026-08-01,11:08,Salão,Refrigerante Lata,Bebidas,1,6.5,6.5,Cartão de Crédito
7,1049,2026-08-01,11:13,Delivery iFood,Feijoada (sábado),Pratos,1,39.0,39.0,iFood Online
8,1049,2026-08-01,11:13,Delivery iFood,Porção de Batata Frita,Porções,1,32.0,32.0,iFood Online
9,1049,2026-08-01,11:13,Delivery iFood,Parmegiana de Carne,Pratos,1,44.0,44.0,iFood Online


### 1. Quantas linhas e colunas tem a tabela?

`df.shape` é um **atributo** (por isso vai sem parênteses) que retorna uma tupla no formato `(linhas, colunas)`.

In [48]:
df.shape

(8506, 10)

A tabela tem **8.506 linhas** e **10 colunas**.

### 2. Qual é o tipo de cada coluna?

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8506 entries, 0 to 8505
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_pedido        8506 non-null   int64  
 1   data             8506 non-null   str    
 2   hora             8506 non-null   str    
 3   canal            8506 non-null   str    
 4   item             8506 non-null   str    
 5   categoria        8506 non-null   str    
 6   quantidade       8506 non-null   int64  
 7   preco_unitario   8506 non-null   float64
 8   valor_total      8506 non-null   float64
 9   forma_pagamento  8491 non-null   str    
dtypes: float64(2), int64(2), str(6)
memory usage: 664.7 KB


**Observações:**

- As colunas `data` e `hora` vieram como **texto** (`str`), e não como data e hora. Nesse formato o pandas não consegue extrair o mês, o dia da semana ou a hora para as análises. Precisarei converter na limpeza.
- A coluna `forma_pagamento` tem **8.491** valores preenchidos, contra **8.506** linhas na tabela: faltam **15 valores** (o equivalente ao `NULL` do SQL). As outras colunas estão completas. Decidir na parte da limpeza

### 3. Quantos pedidos diferentes existem?

In [50]:
df["id_pedido"]

0       1068
1       1054
2       1054
3       1054
4       1012
        ... 
8501    5198
8502    5191
8503    5191
8504    5191
8505    5191
Name: id_pedido, Length: 8506, dtype: int64

- Para selecionar uma coluna uso `nome_da_tabela["nome_da_coluna"] `, no caso uso `df["id_pedido"]`

In [51]:
df.nunique()

id_pedido          4243
data                 79
hora                555
canal                 3
item                 34
categoria             4
quantidade            2
preco_unitario       17
valor_total          30
forma_pagamento       5
dtype: int64

In [52]:
df["id_pedido"].nunique()

4243

- No SQL seria o equivalente à `SELECT COUNT(DISTINCT id_pedido) FROM vendas`;

- No pandas eu primeiro seleciono a coluna com o `df["id_pedido"]` e depois utilizo o método `.nunique()`, que seria equivalente ao ` COUNT(DISTINCT ...)`

- Foram feitos 4243 pedidos

Foram feitos **4.243 pedidos** no período.

### 4. Qual foi o faturamento total do período?

In [53]:
df["valor_total"].sum()

np.float64(205868.0)

- R$ 205.868,00

- É um valor preliminar, antes da limpeza. Se a limpeza mudar alguma linha da tabela, esse número pode mudar

### 5. Por que o número de linhas é diferente do número de pedidos?

- Cada linha representa um **item** vendido, e cada pedido pode ter mais de 1 item.
- Por isso o `id_pedido` se repete: todas as linhas de um mesmo pedido têm o mesmo id. Por exemplo, o pedido **1054** ocupa **3 linhas**: 1 Feijoada (sábado), 2 Marmitex Tradicional e 2 Refrigerantes Lata.
- Isso muda o resultado da análise. Dividindo o faturamento pelo número de **pedidos**, o ticket médio é de **R$ 48,52**. Se as linhas fossem contadas como pedidos, o ticket cairia para **R$ 24,20**, cerca de **metade** do valor real, porque cada pedido tem em média 2 itens (8.506 linhas ÷ 4.243 pedidos ≈ 2).

In [54]:
# Ticket médio correto: faturamento dividido pelo número de pedidos distintos
round(df["valor_total"].sum() / df["id_pedido"].nunique(), 2)

np.float64(48.52)

In [55]:
# Ticket médio errado: faturamento dividido pelo número de linhas (itens), e não de pedidos
round(df["valor_total"].sum() / df["id_pedido"].shape[0], 2)

# Também posso usar a função len() do Python: df["valor_total"].sum() / len(df)

np.float64(24.2)

## Etapa 2. Limpeza dos dados

In [56]:
df["item"].nunique()

34

In [57]:
sorted(df["item"].unique())

['Açaí 300ml',
 'Açaí 300ml ',
 'Cerveja Long Neck',
 'Cerveja Long Neck ',
 'Feijoada (sábado)',
 'Feijoada (sábado) ',
 'Marmitex Grande',
 'Marmitex Tradicional',
 'Marmitex Tradicional ',
 'Mousse de Maracujá',
 'Parmegiana de Carne',
 'Parmegiana de Frango',
 'Porção de Batata Frita',
 'Porção de Batata Frita ',
 'Porção de Calabresa',
 'Porção de Mandioca',
 'Pudim',
 'Refrigerante 2L',
 'Refrigerante Lata',
 'Refrigerante Lata ',
 'Salada Completa',
 'Strogonoff de Frango',
 'Suco Natural',
 'Suco Natural ',
 'cerveja long neck',
 'parmegiana de carne',
 'parmegiana de frango',
 'porção de batata frita',
 'pudim',
 'refrigerante 2l',
 'refrigerante lata',
 'suco natural',
 'Água Mineral',
 'Água Mineral ']

`sorted(df["item"].unique())` nos mostra todos os valores únicos do dataframe na coluna "item", e nele nós verificamos que muitos itens estão sendo escritos de formas diferentes, por exemplo  'Refrigerante Lata','Refrigerante Lata ' e 'refrigerante lata'

- Precisamos corrigir essas palavras diferentes porque o computador pensa que são itens distintos, quando na verdade não são 

- No SQL fariamos `TRIM(item) e LOWER(item)`

- No pandas existem métodos equivalentes, mas com um detalhe: os métodos de texto ficam guardados dentro de um "compartimento" chamado .str
    - Esse .str avisa ao panda "trate essa coluna como um texto e aplique esse método em cada valor"

- `TRIM(item) do SQL equivale a df["item"].str.strip() do pandas`

- `LOWER(item) do SQL equivale a df["item"].str.lower() do pandas`

In [58]:
sorted(df["item"].str.strip().str.title().unique())


['Açaí 300Ml',
 'Cerveja Long Neck',
 'Feijoada (Sábado)',
 'Marmitex Grande',
 'Marmitex Tradicional',
 'Mousse De Maracujá',
 'Parmegiana De Carne',
 'Parmegiana De Frango',
 'Porção De Batata Frita',
 'Porção De Calabresa',
 'Porção De Mandioca',
 'Pudim',
 'Refrigerante 2L',
 'Refrigerante Lata',
 'Salada Completa',
 'Strogonoff De Frango',
 'Suco Natural',
 'Água Mineral']

Usar o `.str.title()` deixa todas as primeiras letras maiúsculas, porém altera tambem o "de" e o "ml".

In [59]:
sorted(df["item"].str.strip().unique())

['Açaí 300ml',
 'Cerveja Long Neck',
 'Feijoada (sábado)',
 'Marmitex Grande',
 'Marmitex Tradicional',
 'Mousse de Maracujá',
 'Parmegiana de Carne',
 'Parmegiana de Frango',
 'Porção de Batata Frita',
 'Porção de Calabresa',
 'Porção de Mandioca',
 'Pudim',
 'Refrigerante 2L',
 'Refrigerante Lata',
 'Salada Completa',
 'Strogonoff de Frango',
 'Suco Natural',
 'cerveja long neck',
 'parmegiana de carne',
 'parmegiana de frango',
 'porção de batata frita',
 'pudim',
 'refrigerante 2l',
 'refrigerante lata',
 'suco natural',
 'Água Mineral']

- Usando apenas o strip() sem o title() nós chegamos ao resultado sem os espaços antes da " que tinhamos no começo, mas agora existem itens que aparecem totalmente em letras minúsculas. 

- Para elas você escreve um dicionário do Python dizendo "esta versão errada vira esta versão certa". Por exemplo: "refrigerante lata": "Refrigerante Lata"

- O método `.replace()`, que recebe o dicionário e troca cada valor que aparecer como chave pelo seu valor


In [60]:
# correcoes = {
#     "versão errada": "Versão Certa",
#     "outra errada": "Outra Certa",
# }

In [61]:
correcoes = {
    "cerveja long neck" : "Cerveja Long Neck",
    "parmegiana de carne" : "Parmegiana de Carne",
    "parmegiana de frango" : "Parmegiana de Frango",
    "porção de batata frita" : "Porção de Batata Frita",
    "pudim" : "Pudim",
    "refrigerante 2l" : "Refrigerante 2L",
    "refrigerante lata" : "Refrigerante Lata",
    "suco natural" : "Suco Natural",   
}

df["item"] = df["item"].str.strip().replace(correcoes)

#### Como funciona a linha `df["item"] = df["item"].str.strip().replace(correcoes)`

O Python executa primeiro o lado **direito** do `=` e só depois guarda o resultado no lado **esquerdo**.

**Lado direito (o cálculo da coluna nova):**

| Pedaço | O que faz |
|---|---|
| `df["item"]` | pega a coluna `item` **atual**, ainda com os nomes errados (8.506 valores) |
| `.str` | avisa ao pandas para tratar a coluna como **texto** e aplicar o próximo método em cada valor |
| `.strip()` | remove os espaços sobrando no início e no fim de cada nome (`'Refrigerante Lata '` vira `'Refrigerante Lata'`) |
| `.replace(correcoes)` | troca cada nome que aparece como **chave** no dicionário pelo seu **valor** (`'refrigerante lata'` vira `'Refrigerante Lata'`). Os nomes que não estão no dicionário continuam iguais |

O resultado continua com **8.506 valores**, cada um na mesma posição de antes, só que corrigido.

**O sinal `=`:** significa "guarde o resultado da direita no lugar indicado à esquerda".

**Lado esquerdo (onde o resultado é guardado):**

- `df["item"]` indica a coluna `item` da tabela. Como ela já existe, a versão antiga é **substituída** pela corrigida.

**Antes e depois:**

| | Antes | Depois |
|---|---|---|
| Nomes distintos em `item` | 34 | 18 |
| Linhas da tabela | 8.506 | 8.506 |

**Equivalente em SQL:**

```sql
UPDATE vendas
SET item = CASE TRIM(item)
               WHEN 'refrigerante lata' THEN 'Refrigerante Lata'
               -- ... uma linha para cada correção
               ELSE TRIM(item)
           END;
```

**Atenção:** o `unique()` e o `sorted()` não podem entrar nesta linha. Eles **resumem** a coluna em 18 valores, e a tabela tem 8.506 linhas, então o pandas daria erro (`Length of values (18) does not match length of index (8506)`).

### ✅ Problema 1 resolvido: nomes de itens escritos de formas diferentes

**O que encontrei:** a coluna `item` tinha **34** nomes distintos, mas o cardápio tem **18** itens. O mesmo item aparecia escrito de formas diferentes, por exemplo `'Refrigerante Lata'`, `'Refrigerante Lata '` (com espaço no final) e `'refrigerante lata'` (tudo em minúsculo). Para o computador, eram itens diferentes, e as vendas de um mesmo produto ficariam divididas. Isso daria resultados errados nas perguntas de negócio que somam por item (itens mais vendidos e curva ABC).

**O que fiz:**
1. `strip()` para remover os espaços sobrando, uma regra geral que vale para todos os nomes;
2. um dicionário com as **8** versões em minúsculo e seus nomes corretos, aplicado com `replace()`.

**Por que não usei o `title()`:** ele resolveria o agrupamento, mas estragaria os nomes: "Porção **D**e Calabresa", "Açaí 300**M**l" e "Feijoada (**S**ábado)". Como eram só 8 correções, o dicionário manteve os nomes exatamente como no cardápio.

**Verificação:** depois da correção, a coluna `item` tem **18** nomes distintos, a tabela continua com 8.506 linhas e o faturamento total continua **R$ 205.868,00**. Só os nomes mudaram, nenhum valor de venda.

In [62]:
# Verificação dos dados alterados e corrigidos do df

df["item"].nunique()

# Agora verificamos realmente a existência de 18 itens distintos igual previsto

18

In [63]:
df["valor_total"].sum()

# Verificamos que o valor de faturamento não foi alterada, confirmando estar correta as mudanças sem alterar o valor das vendas

np.float64(205868.0)

### Problema 2: Linhas duplicadas

**O que encontrei:** Caso o sistema fique lento, problemas de internet, etc. o vendedor pode clicar 2 ou mais vezes para registrar o produto, então podemos encontrar produtos repetidos.

- Um registro duplicado é uma **linha** (um item) idêntica a outra em todas as colunas. Não é o pedido inteiro que se repete, e sim um item dentro dele.

- Por que linhas idênticas são erro, e não uma venda legítima: se o cliente pedisse 2 unidades do mesmo item, o sistema registraria **uma** linha com `quantidade` igual a 2, e não duas linhas iguais.

- Para isso podemos usar o método do pandas .duplicated(), que vai resultar em True ou False

- Como sabemos que True equivale a 1, e False equivale a 0, podemos colocar o método .sum() que irá somar os valores True e nos retornar a quantidade de linhas duplicadas

In [64]:
df.duplicated().sum()

np.int64(6)

In [65]:
df[df.duplicated(keep=False)]

,id_pedido,data,hora,canal,item,categoria,quantidade,preco_unitario,valor_total,forma_pagamento
1474,1770,2026-08-19,12:04,Delivery iFood,Marmitex Tradicional,Pratos,1,22.0,22.0,iFood Online
1477,1770,2026-08-19,12:04,Delivery iFood,Marmitex Tradicional,Pratos,1,22.0,22.0,iFood Online
2291,2170,2026-08-28,11:30,Salão,Refrigerante Lata,Bebidas,1,6.5,6.5,Pix
2293,2170,2026-08-28,11:30,Salão,Refrigerante Lata,Bebidas,1,6.5,6.5,Pix
2688,2361,2026-09-01,12:19,Salão,Refrigerante Lata,Bebidas,1,6.5,6.5,Cartão de Débito
2689,2361,2026-09-01,12:19,Salão,Refrigerante Lata,Bebidas,1,6.5,6.5,Cartão de Débito
3049,2504,2026-09-04,13:01,Balcão,Marmitex Grande,Pratos,2,28.0,56.0,Dinheiro
3050,2504,2026-09-04,13:01,Balcão,Marmitex Grande,Pratos,2,28.0,56.0,Dinheiro
4482,3204,2026-09-19,12:48,Balcão,Marmitex Tradicional,Pratos,1,22.0,22.0,Pix
4484,3204,2026-09-19,12:48,Balcão,Marmitex Tradicional,Pratos,1,22.0,22.0,Pix


In [66]:
df[df.duplicated()]["valor_total"].sum()

# Esse é o valor que as linhas repetidas aumentavam no faturamento total (123 reais)

# Precisamos então remover essas linhas que estão repetidas

np.float64(123.0)

In [67]:
df = df.drop_duplicates()

In [69]:
df.shape


(8500, 10)

In [70]:
df.duplicated().sum()

np.int64(0)

In [71]:
df["valor_total"].sum()

np.float64(205745.0)

In [72]:
df["id_pedido"].nunique()

4243

### ✅ Problema 2 resolvido: linhas duplicadas

**O que encontrei:** **6 linhas** eram cópias idênticas de outras, em todas as colunas. Os pares aparecem com índices vizinhos (por exemplo, 1474 e 1477), o que combina com um registro feito duas vezes seguidas, como um clique duplo quando o sistema trava.

**Por que é erro:** neste dataset, duas unidades do mesmo item são registradas numa única linha, com `quantidade` igual a 2. Então duas linhas idênticas não representam uma venda real, e sim um registro repetido.

**Impacto:** as cópias inflavam o faturamento em **R$ 123,00**. É pouco no total (cerca de 0,06%), mas são vendas que não aconteceram, então não devem entrar no relatório.

**O que fiz:** removi as cópias com `df = df.drop_duplicates()`, que mantém a primeira linha de cada par e descarta a repetição.

**Decisão diferente da forma de pagamento:** as 15 linhas sem forma de pagamento serão **mantidas**, porque são vendas reais com uma informação faltando. Já as duplicatas são **removidas**, porque a venda não aconteceu duas vezes. O critério é o que a linha representa no mundo real.

**Verificação:**

| | Antes | Depois |
|---|---|---|
| Linhas | 8.506 | 8.500 |
| Linhas duplicadas | 6 | 0 |
| Faturamento | R$ 205.868,00 | R$ 205.745,00 |
| Pedidos distintos | 4.243 | 4.243 |

O número de pedidos não mudou porque cada cópia tinha o mesmo `id_pedido` da sua linha original. A duplicata era de um item dentro do pedido, e não do pedido inteiro, então o `nunique()` já contava aquele pedido uma única vez.